In [2]:
import pandas as pd
import seaborn as sns
import scipy.stats as stats
import numpy as np
import matplotlib.pyplot as plt


pd.set_option('display.max_columns',50)

df_imp=pd.read_csv('Import.csv',sep=";",index_col=False,skip_blank_lines=True,quotechar='"',doublequote=True,on_bad_lines="skip",decimal=',',float_precision='round_trip')
df_exp=pd.read_csv('Export.csv',sep=";",index_col=False,skip_blank_lines=True,quotechar='"',doublequote=True,on_bad_lines="skip",decimal=',',float_precision='round_trip')


Two things i'll need to do:

1) Check if i have the data in the 2022<=>2025 timeframe as mentioned in the document
2) check enough info i need to know about my dataset (df.info())

In [3]:
print("\n############# Check Years ###############\n")

def check_years(df:pd.Series):
    df_to_list_sorted=df.unique().tolist()
    df_to_list_sorted.sort()
    return df_to_list_sorted

print(check_years(df_exp['Année']))
print(check_years(df_imp['Année']))

print("\n############# Info #############\n")

display(df_exp.info())
display(df_imp.info())


############# Check Years ###############

[2022, 2023, 2024, 2025]
[2022, 2023, 2024, 2025]

############# Info #############

<class 'pandas.DataFrame'>
RangeIndex: 12821 entries, 0 to 12820
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Code du chapitre SH                       12821 non-null  int64  
 1   Libellé du chapitre SH                    12821 non-null  str    
 2   Code du produit SH                        12821 non-null  int64  
 3   Libellé du produit SH                     12821 non-null  str    
 4   Code du groupement d'utilisation          12821 non-null  int64  
 5   Libellé du groupement d'utilisation       12821 non-null  str    
 6   Code du nouveau produits remarquables     12821 non-null  int64  
 7   Libellé du nouveau produits remarquables  12821 non-null  str    
 8   Code de la section CTCI                   12821 non-null  

None

<class 'pandas.DataFrame'>
RangeIndex: 64428 entries, 0 to 64427
Data columns (total 24 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Code du chapitre SH                       64428 non-null  int64  
 1   Libellé du chapitre SH                    64428 non-null  str    
 2   Code du produit SH                        64428 non-null  int64  
 3   Libellé du produit SH                     64428 non-null  str    
 4   Code du groupement d'utilisation          64428 non-null  int64  
 5   Libellé du groupement d'utilisation       64428 non-null  str    
 6   Code du nouveau produits remarquables     64428 non-null  int64  
 7   Libellé du nouveau produits remarquables  64428 non-null  str    
 8   Code de la section CTCI                   64428 non-null  int64  
 9   Libellé de la section CTCI                64428 non-null  str    
 10  Code de la division CTCI                  644

None

For Import dataframe:
    * 3 columns have 1 null each, both have the type str so i'll fill it with "unknown"

For Export dataframe:
    * 1 columns have 1 null each, it has the type str so i'll fill it with "unknown"

we need to do the following:

1) drop the columns that have no analysis significance
2) Check cardinality

In [19]:
print('################ drop unuseful columns')

cols_drop_exp=[col for col in df_exp.columns if 'code' in col.lower() and 'produit sh' not in col.lower() ]
cols_drop_imp=[col for col in df_imp.columns if 'code' in col.lower() and 'produit sh' not in col.lower() ]

df_exp=df_exp.drop(cols_drop_exp,axis=1)
df_imp=df_imp.drop(cols_drop_imp,axis=1)
display(df_exp.columns,df_imp.columns)

################ drop unuseful columns


Index(['Libellé du chapitre SH', 'Code du produit SH', 'Libellé du produit SH',
       'Libellé du groupement d'utilisation',
       'Libellé du nouveau produits remarquables',
       'Libellé de la section CTCI', 'Libellé de la division CTCI',
       'Libellé du groupe CTCI', 'Continent', 'Libellé du pays',
       'Libellé du flux', 'Année', 'Mois', 'Poids en KG', 'Valeur DHS',
       'Unité complementaire'],
      dtype='str')

Index(['Libellé du chapitre SH', 'Code du produit SH', 'Libellé du produit SH',
       'Libellé du groupement d'utilisation',
       'Libellé du nouveau produits remarquables',
       'Libellé de la section CTCI', 'Libellé de la division CTCI',
       'Libellé du groupe CTCI', 'Continent', 'Libellé du pays',
       'Libellé du flux', 'Année', 'Mois', 'Poids en KG', 'Valeur DHS',
       'Unité complementaire'],
      dtype='str')

In [ ]:
print('################ Checking Cardinality')
## value count on each column?

def cardinality_check(df_name:str,df:pd.DataFrame):
    print(f"Checking Cardinality on {df_name}:\n")
    for col in df.columns:
        print(f"{col:<50}: Cardinality of {len(df[col].unique()):<10} data type : {df[col].dtype}")
    print(f'\n{30*"#"} Finished {30*"#"}\n')

cardinality_check('export',df_exp)
cardinality_check('import',df_imp)

################ Checking Cardinality
Checking Cardinality on export:

Libellé du chapitre SH                            : Cardinality of 1          data type : str
Code du produit SH                                : Cardinality of 237        data type : int64
Libellé du produit SH                             : Cardinality of 236        data type : str
Libellé du groupement d'utilisation               : Cardinality of 3          data type : str
Libellé du nouveau produits remarquables          : Cardinality of 9          data type : str
Libellé de la section CTCI                        : Cardinality of 2          data type : str
Libellé de la division CTCI                       : Cardinality of 4          data type : str
Libellé du groupe CTCI                            : Cardinality of 9          data type : str
Continent                                         : Cardinality of 7          data type : str
Libellé du pays                                   : Cardinality of 143        dat

elements that have the least value_count will be turned into other, so that i can have less cardinality

think through whether it's safe to minimize the cardinality of products since they might need to check

also i'll need to groupby libellé du produit sh & order by the count of elements inside the groupby

In [73]:
## since the code column is higher than the 

df_exp_check=df_exp.groupby(by='Libellé du produit SH',as_index=True)['Code du produit SH'].nunique()
df_imp_check=df_imp.groupby(by='Libellé du produit SH',as_index=True)['Code du produit SH'].nunique()
x=df_exp_check[df_exp_check>1]
y=df_imp_check[df_imp_check>1]
print(x,y,sep="\n\n########\n\n")

Libellé du produit SH
AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3    2
Name: Code du produit SH, dtype: int64

########

Libellé du produit SH
AUTRES PARTIES ET ACCESSOIRES DE MOTOCYCLES                                                                               2
AUTRES TRACTEURS NEUFS, PUISSANCE ENTRE 37 KW ET 75KW                                                                     2
AUTRES VEHICULES                                                                                                          2
AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3                                                                  2
AUTRES VEHICULES NEUFS,A MOTEUR A PISTON ALTERN.CYL<=1000CM3                                                              2
BROUETTES                                                                                                                 2
CHASSIS AVEC CABINE PR TRANSPORT MARCHANDISES,  MOT. A PISTON ALLUM. PAR COMPRESSION ET ELECTRIQUE, 

In [74]:
# For export
df_exp_codes = df_exp.groupby('Libellé du produit SH')['Code du produit SH'].unique()
print("Export duplicates and their codes:")
print(df_exp_codes[df_exp_codes.str.len() > 1])

print("\n" + "#"*40 + "\n")

# For import
df_imp_codes = df_imp.groupby('Libellé du produit SH')['Code du produit SH'].unique()
print("Import duplicates and their codes:")
print(df_imp_codes[df_imp_codes.str.len() > 1])

Export duplicates and their codes:
Libellé du produit SH
AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3    [8703238600, 8703238400]
Name: Code du produit SH, dtype: object

########################################

Import duplicates and their codes:
Libellé du produit SH
AUTRES PARTIES ET ACCESSOIRES DE MOTOCYCLES                                                                               [8714100090, 8714109019]
AUTRES TRACTEURS NEUFS, PUISSANCE ENTRE 37 KW ET 75KW                                                                     [8701934091, 8701939010]
AUTRES VEHICULES                                                                                                          [8716800099, 8716809990]
AUTRES VEHICULES NEUFS, CYLINDREE ENTRE 1500 ET 3000 CM3                                                                  [8703238400, 8703238600]
AUTRES VEHICULES NEUFS,A MOTEUR A PISTON ALTERN.CYL<=1000CM3                                                              [8703218